# 03. Exportación y preparación para Power BI

**Responsable:** Linda Morales — Power BI e informe final.

Este notebook continúa el flujo de `02_modelo_predictivo.ipynb`. No vuelve a entrenar el modelo: carga y valida sus salidas livianas, crea una tabla auxiliar en formato largo y documenta los campos que alimentarán `powerbi/dashboard_mundial_2026.pbix`.

Fuentes oficiales para el dashboard:

- `outputs_powerbi/team_probabilities.csv`
- `outputs_powerbi/predicted_podium.csv`
- `outputs_powerbi/model_metrics.csv`
- `outputs_powerbi/model_metadata.json`

In [13]:
from pathlib import Path
import json

import pandas as pd

# Funciona al ejecutar desde notebooks/ o desde la raíz del proyecto.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUTPUTS = ROOT / 'outputs_powerbi'

input_paths = {
    'probabilidades': OUTPUTS / 'team_probabilities.csv',
    'podio': OUTPUTS / 'predicted_podium.csv',
    'metricas': OUTPUTS / 'model_metrics.csv',
    'metadatos': OUTPUTS / 'model_metadata.json',
}

missing_files = [str(path) for path in input_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('Faltan archivos requeridos:\n' + '\n'.join(missing_files))

team = pd.read_csv(input_paths['probabilidades'])
podium = pd.read_csv(input_paths['podio'])
metrics = pd.read_csv(input_paths['metricas'])
with open(input_paths['metadatos'], encoding='utf-8') as archivo:
    metadata = json.load(archivo)

print(f'Carpeta del proyecto: {ROOT}')
print(f'Selecciones cargadas: {len(team)}')
print(f"Modelo seleccionado: {metadata['modelo_seleccionado']}")
print(f"Simulaciones: {metadata['n_simulaciones']:,}")

Carpeta del proyecto: c:\Users\esmed\Downloads\Proyecto_Mineria_Avanzada-main\Proyecto_Mineria_Avanzada-main
Selecciones cargadas: 48
Modelo seleccionado: gradient_boosting
Simulaciones: 5,000


## Diccionario de datos

La tabla principal contiene una fila por selección. Las columnas `p_*` son probabilidades expresadas entre 0 y 1.

| Campo | Uso en Power BI |
|---|---|
| `team` | Selección |
| `p_round32` | Probabilidad de avanzar a ronda de 32 |
| `p_round16` | Probabilidad de avanzar a octavos |
| `p_quarterfinal` | Probabilidad de avanzar a cuartos |
| `p_semifinal` | Probabilidad de avanzar a semifinal |
| `p_final` | Probabilidad de avanzar a la final |
| `p_champion` | Probabilidad de ser campeón |
| `p_runner_up` | Probabilidad de ser subcampeón |
| `p_third_place` | Probabilidad de obtener tercer lugar |
| `p_fourth_place` | Probabilidad de obtener cuarto lugar |
| `fifa_rank`, `fifa_points` | Posición y puntos FIFA |
| `elo_rating` | Valoración Elo |
| `confederation` | Confederación del equipo |

In [14]:
required_team_columns = [
    'team', 'p_round32', 'p_round16', 'p_quarterfinal',
    'p_semifinal', 'p_final', 'p_champion', 'p_runner_up',
    'p_third_place', 'p_fourth_place', 'fifa_rank',
    'fifa_points', 'elo_rating', 'confederation',
]
probability_columns = [column for column in required_team_columns if column.startswith('p_')]
required_podium_columns = ['position', 'label', 'team', 'probability', 'fifa_rank', 'elo_rating']
required_metric_columns = ['modelo', 'log_loss', 'accuracy']

assert set(required_team_columns).issubset(team.columns), 'Faltan columnas en team_probabilities.csv'
assert set(required_podium_columns).issubset(podium.columns), 'Faltan columnas en predicted_podium.csv'
assert set(required_metric_columns).issubset(metrics.columns), 'Faltan columnas en model_metrics.csv'
assert len(team) == 48, f'Se esperaban 48 selecciones y se encontraron {len(team)}'
assert team['team'].is_unique, 'Hay selecciones duplicadas'
assert team[required_team_columns].notna().all().all(), 'Hay valores vacíos en la tabla principal'
assert team[probability_columns].ge(0).all().all(), 'Hay probabilidades menores que 0'
assert team[probability_columns].le(1).all().all(), 'Hay probabilidades mayores que 1'
assert set(podium['position']) == {1, 2, 3, 4}, 'El podio no contiene las cuatro posiciones'
assert podium['team'].is_unique, 'Hay equipos duplicados en el podio'
assert metadata['modelo_seleccionado'] in set(metrics['modelo']), 'El modelo seleccionado no aparece en las métricas'

selected_model = metrics.loc[metrics['modelo'] == metadata['modelo_seleccionado']].iloc[0]
print('Validación terminada correctamente.')
print(f"Accuracy: {selected_model['accuracy']:.2%}")
print(f"Log loss: {selected_model['log_loss']:.6f}")

Validación terminada correctamente.
Accuracy: 60.91%
Log loss: 0.874594


In [15]:
round_names = {
    'p_round32': 'Ronda de 32',
    'p_round16': 'Octavos',
    'p_quarterfinal': 'Cuartos de final',
    'p_semifinal': 'Semifinal',
    'p_final': 'Final',
    'p_champion': 'Campeón',
}
round_order = {code: order for order, code in enumerate(round_names, start=1)}

team_probabilities_long = team.melt(
    id_vars=['team', 'fifa_rank', 'fifa_points', 'elo_rating', 'confederation'],
    value_vars=list(round_names),
    var_name='round_code',
    value_name='probability',
)
team_probabilities_long['round'] = team_probabilities_long['round_code'].map(round_names)
team_probabilities_long['round_order'] = team_probabilities_long['round_code'].map(round_order)
team_probabilities_long = team_probabilities_long.sort_values(['team', 'round_order']).reset_index(drop=True)

long_path = OUTPUTS / 'team_probabilities_long.csv'
team_probabilities_long.to_csv(long_path, index=False, encoding='utf-8-sig')

assert len(team_probabilities_long) == len(team) * len(round_names)
print(f'Archivo creado: {long_path}')
print(f'Filas exportadas: {len(team_probabilities_long)}')
display(team_probabilities_long.head(12))

Archivo creado: c:\Users\esmed\Downloads\Proyecto_Mineria_Avanzada-main\Proyecto_Mineria_Avanzada-main\outputs_powerbi\team_probabilities_long.csv
Filas exportadas: 288


,team,fifa_rank,fifa_points,elo_rating,confederation,round_code,probability,round,round_order
0,Algeria,29,1576.803,1726.0,CAF,p_round32,0.6268,Ronda de 32,1
1,Algeria,29,1576.803,1726.0,CAF,p_round16,0.2950,Octavos,2
2,Algeria,29,1576.803,1726.0,CAF,p_quarterfinal,0.1314,Cuartos de final,3
3,Algeria,29,1576.803,1726.0,CAF,p_semifinal,0.0556,Semifinal,4
4,Algeria,29,1576.803,1726.0,CAF,p_final,0.0186,Final,5
5,Algeria,29,1576.803,1726.0,CAF,p_champion,0.0050,Campeón,6
6,Argentina,2,1925.149,2113.0,CONMEBOL,p_round32,0.9890,Ronda de 32,1
7,Argentina,2,1925.149,2113.0,CONMEBOL,p_round16,0.6154,Octavos,2
8,Argentina,2,1925.149,2113.0,CONMEBOL,p_quarterfinal,0.4220,Cuartos de final,3
9,Argentina,2,1925.149,2113.0,CONMEBOL,p_semifinal,0.2854,Semifinal,4


In [16]:
podium_report = podium[['position', 'label', 'team', 'probability']].copy()
podium_report['probability'] = podium_report['probability'].map(lambda value: f'{value:.2%}')
metrics_report = metrics.copy()
metrics_report['accuracy'] = metrics_report['accuracy'].map(lambda value: f'{value:.2%}')
metrics_report['log_loss'] = metrics_report['log_loss'].map(lambda value: f'{value:.6f}')

print('Podio proyectado')
display(podium_report)
print('Comparación de modelos')
display(metrics_report)

Podio proyectado


,position,label,team,probability
0,1,Campeon,Argentina,14.00%
1,2,Subcampeon,Portugal,5.42%
2,3,Tercer lugar,Spain,6.50%
3,4,Cuarto lugar,Norway,4.14%


Comparación de modelos


,modelo,log_loss,accuracy
0,gradient_boosting,0.874594,60.91%
1,random_forest,0.899578,58.42%
2,regresion_logistica,0.927024,55.30%


In [ ]:
# Versiones compatibles con Power BI configurado en español.
# Se usa punto y coma como separador para no confundirlo con la coma decimal.
powerbi_es_exports = {
    'team_probabilities_powerbi_es.csv': team,
    'team_probabilities_long_powerbi_es.csv': team_probabilities_long,
    'predicted_podium_powerbi_es.csv': podium,
    'model_metrics_powerbi_es.csv': metrics,
}

for filename, dataframe in powerbi_es_exports.items():
    output_path = OUTPUTS / filename
    dataframe.to_csv(
        output_path,
        index=False,
        sep=';',
        decimal=',',
        encoding='utf-8-sig',
    )
    print(f'Archivo Power BI ES creado: {output_path}')

## Diseño requerido del dashboard

El README propone cuatro vistas:

1. **Resumen del podio:** tarjetas o tabla con `predicted_podium.csv`.
2. **Probabilidades por selección:** ranking de `team_probabilities.csv`, ordenado de mayor a menor por `p_champion`, con filtros de equipo y confederación.
3. **Camino al título:** gráfico de líneas o barras con `team_probabilities_long.csv`; usar `round` como eje, `probability` como valor, `team` como leyenda/filtro y ordenar `round` mediante `round_order`.
4. **Metodología:** tabla comparativa de `model_metrics.csv` y notas de `model_metadata.json` sobre el modelo seleccionado, simulaciones y escenario.

En Power BI, las columnas de probabilidad y `accuracy` deben formatearse como porcentaje. Para `log_loss`, un valor menor es mejor; para `accuracy`, un valor mayor es mejor.

## Nota para el informe final

El modelo seleccionado es **Gradient Boosting** y los resultados se obtuvieron mediante 5,000 simulaciones. Las probabilidades representan estimaciones del modelo, no resultados garantizados. El corte de datos declarado por el proyecto es el **8 de julio de 2026**.

Después de ejecutar este notebook, importar en Power BI:

- `team_probabilities_powerbi_es.csv`
- `team_probabilities_long_powerbi_es.csv`
- `predicted_podium_powerbi_es.csv`
- `model_metrics_powerbi_es.csv`

Estas versiones usan `;` como separador de columnas y `,` como separador decimal. Usar `model_metadata.json` como respaldo para redactar la página metodológica.